# Perfusion Operating Mode Test Notebook

This notebook tests the **PERFUSION** operating mode implementation.

**Operating Mode Characteristics:**
- Starts directly in perfusion mode (no fed-batch phase)
- Cell retention with continuous medium exchange
- Constant volume operation with feed and harvest streams

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Perfusion-specific validations (cell retention, volume control, perfusion strategies)

**Note:** For modes that start with a fed-batch phase before perfusion, see:
- Concentrated Fed-Batch: Fed-batch → Perfusion (triggered by metabolite levels)
- Continuous Perfusion: Fed-batch → Continuous Perfusion (with continuous feed)

## 1. Setup and Imports

In [14]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [15]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "perfusion"])
#params.set_parameter_value('process_schedule.t_max', 200)  # shorten simulation time for testing

# Initialize simulation with PERFUSION mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.PERFUSION,
    perfusion_strategy=PerfusionStrategy.CONSTANT,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False  # Store detailed data for testing
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Perfusion Strategy: {sim.perfusion_strategy}")
print(f"  Gas Controller: {sim.gas_controller}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\perfusion.yaml
Successfully loaded parameters from 5 file(s).
✓ Simulation initialized
  Operating Mode: P

## 4. Multi-Batch Run (Auto-Extracted Initial Conditions)

In [16]:

# Run 5 repeated batches

num_batches = 5
multi_result = sim.run_simulation(number_of_batches=num_batches)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L, V={batch.V_final:.2f} L")


run_simulation execution time: 286.413 ms
✓ 5 batches completed

Summary Statistics:
  Average batch duration: 263.49 h
  STY: 0.472 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 357.47 h, Xv=1.41e+11 cells/L, V=600.00 L
  Batch 2: 270.54 h, Xv=1.41e+11 cells/L, V=600.00 L
  Batch 3: 270.54 h, Xv=1.41e+11 cells/L, V=600.00 L
  Batch 4: 61.44 h, Xv=3.35e+10 cells/L, V=600.00 L
  Batch 5: 357.47 h, Xv=1.41e+11 cells/L, V=600.00 L


## 5. Data Integrity Validation

In [17]:

# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("\n1. State Variables:")
# y_points shape: (n_states, n_timepoints) - states are rows
# [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA, C_Osm_bg]
state_vars = ['Xv', 'C_Glc', 'C_Gln', 'C_Lac', 'C_Amm', 'V', 'C_O2', 'C_CT', 'C_H', 'C_Na', 'C_AA', 'C_Osm_bg']
for idx, var in enumerate(state_vars):
    data = np.asarray(batch_1.y_points[idx], dtype=float)  # Ensure numeric array
    print(f"  {var}: {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_AA', 'q_O2', 'q_CO2']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n7. Dilution Rate (Perfusion-specific):")
D_list = batch_1.D_list
print(f"  D: {len(D_list)} points, range [{D_list.min():.4f}, {D_list.max():.4f}] 1/h")
assert len(D_list) > 0, "D_list is empty"
assert not np.any(np.isnan(D_list)), "D_list contains NaN values"

print("\n✓ All data integrity checks passed")

Data Integrity Checks for Batch 1:

1. State Variables:
  Xv: 427 points, range [6.00e+08, 1.41e+11]
  C_Glc: 427 points, range [3.15e+00, 2.50e+01]
  C_Gln: 427 points, range [5.71e-02, 1.00e+01]
  C_Lac: 427 points, range [0.00e+00, 9.82e+00]
  C_Amm: 427 points, range [0.00e+00, 3.24e+00]
  V: 427 points, range [6.00e+02, 6.00e+02]
  C_O2: 427 points, range [1.00e-02, 1.00e-02]
  C_CT: 427 points, range [1.78e+01, 2.90e+01]
  C_H: 427 points, range [6.31e-08, 6.31e-08]
  C_Na: 427 points, range [0.00e+00, 0.00e+00]
  C_AA: 427 points, range [5.00e+00, 9.31e+00]
  C_Osm_bg: 427 points, range [2.50e+02, 2.50e+02]

2. Kinetic Rates:
  mu: 427 points, range [4.06e-03, 3.03e-02]
  q_Glc: 427 points, range [7.17e-12, 5.35e-11]
  q_Gln: 427 points, range [2.11e-12, 1.57e-11]
  q_Lac: 427 points, range [4.18e-12, 3.12e-11]
  q_Amm: 427 points, range [1.38e-12, 1.03e-11]
  q_AA: 427 points, range [2.07e-12, 1.55e-11]
  q_O2: 427 points, range [2.07e-11, 1.55e-10]
  q_CO2: 427 points, range [

## 6. Performance Metrics

In [18]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Conversion Efficiency: {multi_result.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {multi_result.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {multi_result.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_conversion_efficiency_m3_kg'] < 1000, "Medium conversion efficiency should be positive and reasonable"
assert multi_result.summary_stats['protein_conversion_efficiency_kg_kg'] > 0, "Protein conversion efficiency should be positive"
assert multi_result.summary_stats['feed_conversion_efficiency_kg_kg'] > 0, "Feed conversion efficiency should be positive"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")


Performance Metrics (5-Batches):

 Total Biomass: 56.19 kg
  STY: 0.472 kg/m3/day
  Medium Conversion Efficiency: 0.970 m3/kg DCM
  Protein Conversion Efficiency: 2.912 kg/kg
  Feed Conversion Efficiency: 5.357 kg/kg
  Total Simulation Time: 1317.47 h
  Number of Batches: 5

✓ Performance metrics validated


## 7. Perfusion-Specific Validations

In [19]:
print("Perfusion-Specific Validations:")

# 1. Volume control (should remain approximately constant after perfusion starts)
print("\n1. Volume Control:")
for i, batch in enumerate(sim.data.batches, 1):
    V_initial = batch.y_points[5, 0]
    V_final = batch.y_points[5, -1]
    V_max = batch.y_points[5, :].max()
    V_mean = batch.y_points[5, :].mean()
    V_std = batch.y_points[5, :].std()
    print(f"  Batch {i}: V_initial={V_initial:.2f} L, V_final={V_final:.2f} L, V_max={V_max:.2f} L")
    print(f"           V_mean={V_mean:.2f} L, V_std={V_std:.3f} L (CV={V_std/V_mean*100:.2f}%)")

# 2. Dilution rate profile (should increase with perfusion strategy)
print("\n2. Dilution Rate Profile (Batch 1):")
D_initial = batch_1.D_list[0]
D_final = batch_1.D_list[-1]
D_max = batch_1.D_list.max()
D_mean = batch_1.D_list.mean()
print(f"  D_initial: {D_initial:.4f} 1/h")
print(f"  D_final: {D_final:.4f} 1/h")
print(f"  D_max: {D_max:.4f} 1/h")
print(f"  D_mean: {D_mean:.4f} 1/h")

if sim.perfusion_strategy in [PerfusionStrategy.TOXICITY_BASED_STEP, PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL]:
    print(f"  Perfusion strategy: {sim.perfusion_strategy}")
    print(f"  Expected: D should increase when toxicity thresholds are reached")

# 3. Cell retention efficiency
print("\n3. Cell Retention and Growth:")
eff_separator = float(params.perfusion.eff_separator)
for i, batch in enumerate(sim.data.batches, 1):
    Xv_initial = batch.y_points[0, 0]
    Xv_final = batch.y_points[0, -1]
    Xv_max = batch.y_points[0, :].max()
    fold_increase = Xv_max / Xv_initial
    print(f"  Batch {i}:")
    print(f"    Xv_initial: {Xv_initial:.2e} cells/L")
    print(f"    Xv_max: {Xv_max:.2e} cells/L")
    print(f"    Xv_final: {Xv_final:.2e} cells/L")
    print(f"    Fold increase: {fold_increase:.2f}x")
    print(f"    Cell retention efficiency: {eff_separator*100:.1f}%")
    assert Xv_max > Xv_initial, f"Batch {i}: No cell growth detected"

# 4. Metabolite control (lactate and ammonia should be controlled)
print("\n4. Metabolite Control:")
C_Lac_trigger = float(params.perfusion.C_Lac_pf_trig) if params.perfusion.C_Lac_pf_trig is not None else float('inf')
C_Amm_trigger = float(params.perfusion.C_Amm_pf_trig) if params.perfusion.C_Amm_pf_trig is not None else float('inf')
for i, batch in enumerate(sim.data.batches, 1):
    C_Lac_max = batch.y_points[3, :].max()
    C_Amm_max = batch.y_points[4, :].max()
    C_Lac_final = batch.y_points[3, -1]
    C_Amm_final = batch.y_points[4, -1]
    print(f"  Batch {i}:")
    print(f"    Lactate: max={C_Lac_max:.2f} mmol/L, final={C_Lac_final:.2f} mmol/L (trigger={C_Lac_trigger:.2f})")
    print(f"    Ammonia: max={C_Amm_max:.2f} mmol/L, final={C_Amm_final:.2f} mmol/L (trigger={C_Amm_trigger:.2f})")
    
    # Check if perfusion was triggered by metabolites
    if C_Lac_max >= C_Lac_trigger:
        print(f"    → Lactate trigger reached")
    if C_Amm_max >= C_Amm_trigger:
        print(f"    → Ammonia trigger reached")

# 5. Perfusion rate increase timing
print("\n5. Perfusion Rate Increase Analysis (Batch 1):")
# Note: In pure PERFUSION mode, perfusion is already active from t=0
# Identify when perfusion rate increases above baseline (D > D_baseline)
D_baseline = float(params.perfusion.D_baseline)
perfusion_increased = batch_1.D_list > (D_baseline * 1.1)  # 10% threshold above baseline
if np.any(perfusion_increased):
    increase_idx = np.argmax(perfusion_increased)
    t_increase = batch_1.t_points[increase_idx]
    Xv_at_increase = batch_1.y_points[0, increase_idx]
    C_Lac_at_increase = batch_1.y_points[3, increase_idx]
    C_Amm_at_increase = batch_1.y_points[4, increase_idx]
    print(f"  Perfusion rate increased above baseline at t={t_increase:.2f} h")
    print(f"  Conditions at rate increase:")
    print(f"    Xv: {Xv_at_increase:.2e} cells/L")
    print(f"    Lactate: {C_Lac_at_increase:.2f} mmol/L")
    print(f"    Ammonia: {C_Amm_at_increase:.2f} mmol/L")
else:
    print(f"  Perfusion rate remained at baseline (no trigger activated)")

# 6. Substrate levels maintenance
print("\n6. Substrate Levels:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Glc_min = batch.y_points[1, :].min()
    C_Gln_min = batch.y_points[2, :].min()
    C_Glc_final = batch.y_points[1, -1]
    C_Gln_final = batch.y_points[2, -1]
    print(f"  Batch {i}:")
    print(f"    Glucose: min={C_Glc_min:.2f} mmol/L, final={C_Glc_final:.2f} mmol/L")
    print(f"    Glutamine: min={C_Gln_min:.2f} mmol/L, final={C_Gln_final:.2f} mmol/L")

print("\n✓ All perfusion-specific validations completed")

Perfusion-Specific Validations:

1. Volume Control:
  Batch 1: V_initial=600.00 L, V_final=600.00 L, V_max=600.00 L
           V_mean=600.00 L, V_std=0.000 L (CV=0.00%)
  Batch 2: V_initial=600.00 L, V_final=600.00 L, V_max=600.00 L
           V_mean=600.00 L, V_std=0.000 L (CV=0.00%)
  Batch 3: V_initial=600.00 L, V_final=600.00 L, V_max=600.00 L
           V_mean=600.00 L, V_std=0.000 L (CV=0.00%)
  Batch 4: V_initial=600.00 L, V_final=600.00 L, V_max=600.00 L
           V_mean=600.00 L, V_std=0.000 L (CV=0.00%)
  Batch 5: V_initial=600.00 L, V_final=600.00 L, V_max=600.00 L
           V_mean=600.00 L, V_std=0.000 L (CV=0.00%)

2. Dilution Rate Profile (Batch 1):
  D_initial: 0.0600 1/h
  D_final: 0.0600 1/h
  D_max: 0.0600 1/h
  D_mean: 0.0600 1/h

3. Cell Retention and Growth:
  Batch 1:
    Xv_initial: 6.00e+08 cells/L
    Xv_max: 1.41e+11 cells/L
    Xv_final: 1.41e+11 cells/L
    Fold increase: 235.00x
    Cell retention efficiency: 99.0%
  Batch 2:
    Xv_initial: 7.05e+09 cell

In [20]:
sim.data.plot_biomass_metabolites()

In [21]:
sim.data.plot_volume_feed()

In [22]:
sim.data.plot_pH_control(batch_number=1)

In [23]:
sim.data.plot_gas_transfer()

In [24]:
sim.data.plot_kinetics()

## 8. Test Different Perfusion Strategies

In [25]:
# Using func_timeout to test different perfusion strategies with a timeout to prevent hanging simulations.
from func_timeout import func_timeout, FunctionTimedOut

print("Testing Different Perfusion Strategies:\n")

# We need to add some timeout to prevent hanging if a strategy causes issues. usually this is because of non-fitting parameters for that strategy, which we want to catch and fix.
TIMEOUT_SECONDS = 5

perfusion_strategies = [
    PerfusionStrategy.CONSTANT,
    PerfusionStrategy.TOXICITY_BASED_STEP,
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL,
    PerfusionStrategy.CSPR,
    PerfusionStrategy.GLUCOSE_SETPOINT,
]
results = {}

params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "perfusion"])

def run_strategy(perf_strat):
    sim_test = CellCultureSimulation(
        parameters=params,
        kinetics=Kinetics.STOICHIOMETRIC,
        operating_mode=OperatingMode.PERFUSION,
        perfusion_strategy=perf_strat,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=False
    )
    result = sim_test.run_simulation()
    batch = result.batches[0]
    return {
        't_end': batch.t_end,
        'Xv_final': batch.Xv_final,
        'V_final': batch.V_final,
        'termination': batch.termination_reason
    }

results = {}

for perf_strat in perfusion_strategies:
    print(f"Testing {perf_strat}...")
    try:
        batch_data = func_timeout(TIMEOUT_SECONDS, run_strategy, args=(perf_strat,))
        results[str(perf_strat)] = batch_data
        print(f"  Duration: {batch_data['t_end']:.2f} h, Xv_final: {batch_data['Xv_final']:.2e} cells/L, V_final: {batch_data['V_final']:.2f} L")
        print(f"  Termination: {batch_data['termination']}")

    except FunctionTimedOut:
        print(f"  ✗ TIMEOUT: {perf_strat} exceeded {TIMEOUT_SECONDS}s limit — skipping.")
        results[str(perf_strat)] = {'error': f'Timeout after {TIMEOUT_SECONDS}s'}

    except Exception as e:
        print(f"  ✗ ERROR in {perf_strat}: {type(e).__name__}: {e}")
        results[str(perf_strat)] = {'error': str(e)}

print("\n✓ All perfusion strategies attempted")

Testing Different Perfusion Strategies:

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\perfusion.yaml
Successfully loaded parameters from 5 file(s).
Tes

## 9. Compare Perfusion Strategies: Inlet Rates and Glucose

In [26]:
# Comparison plot: perfusion profiles and glucose concentration across all perfusion strategies
import copy
from func_timeout import func_timeout, FunctionTimedOut
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from animakinetica.analysis.batch_visualization import (
    plot_batch_volume_feed,
    plot_batch_biomass_metabolites,
)

TIMEOUT_SECONDS = 10

# Use same simulation setup for fair comparison
perfusion_strategies_compare = [
    PerfusionStrategy.CONSTANT,
    PerfusionStrategy.TOXICITY_BASED_STEP,
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL,
    PerfusionStrategy.CSPR,
    PerfusionStrategy.GLUCOSE_SETPOINT,
]

color_map = {
    PerfusionStrategy.CONSTANT: "#1f77b4",
    PerfusionStrategy.TOXICITY_BASED_STEP: "#ff7f0e",
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL: "#2ca02c",
    PerfusionStrategy.CSPR: "#d62728",
    PerfusionStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

fig_compare = make_subplots(
    rows=1, cols=5,
    subplot_titles=(
        "Process Perfusion Rate Comparison",
        "Base Feed Rate Comparison",
        "Total Inlet Rate Comparison",
        "Volume Comparison",
        "Glucose Comparison",
    ),
    horizontal_spacing=0.08
)

params_compare = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "perfusion"])

def run_compare_simulation(perf_strategy, base_params):
    params_test = copy.deepcopy(base_params)
    params_test.update("excess_kinetics_batch")  # Use excess kinetics for more dynamic response in perfusion testing
    sim_test = CellCultureSimulation(
        parameters=params_test,
        kinetics=Kinetics.EXCESS,
        operating_mode=OperatingMode.PERFUSION,
        perfusion_strategy=perf_strategy,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=False,
        solver_method="BDF"
    )
    return sim_test.run_simulation(number_of_batches=1)

for ps in perfusion_strategies_compare:
    print(f"Simulating {ps} for comparison...")
    try:
        res = func_timeout(TIMEOUT_SECONDS, run_compare_simulation, args=(ps, params_compare))
    except FunctionTimedOut:
        print(f"  ✗ TIMEOUT: {ps} exceeded {TIMEOUT_SECONDS}s limit — skipping.")
        continue
    except Exception as e:
        print(f"  ✗ ERROR in {ps}: {type(e).__name__}: {e} — skipping.")
        continue

    # Reuse existing plotting utilities and extract traces from their outputs
    fig_volume = plot_batch_volume_feed(res, batch_number=1, layout_kwargs={"showlegend": False})
    fig_biomass = plot_batch_biomass_metabolites(res, batch_number=1, layout_kwargs={"showlegend": False})

    process_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Process Feed Rate")
    base_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Base Feed Rate")
    total_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Total Inlet Rate")
    volume_trace = next(tr for tr in fig_volume.data if tr.name == "Volume")
    glucose_trace = next(tr for tr in fig_biomass.data if tr.name == "Glucose")

    label = ps.name
    color = color_map.get(ps, None)

    fig_compare.add_trace(
        go.Scatter(
            x=process_rate_trace.x,
            y=process_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=True,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_process: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=1,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=base_rate_trace.x,
            y=base_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_base: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=2,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=total_rate_trace.x,
            y=total_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_total: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=3,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=volume_trace.x,
            y=volume_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color, dash="dot"),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>V: %{y:.2f} L<extra></extra>",
        ),
        row=1, col=4,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=glucose_trace.x,
            y=glucose_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Glucose: %{y:.2f} mmol/L<extra></extra>",
        ),
        row=1, col=5,
    )

for col in range(1, 6):
    fig_compare.update_xaxes(title_text="Time (h)", row=1, col=col)

fig_compare.update_yaxes(title_text="Process Perfusion Rate (L/h)", row=1, col=1)
fig_compare.update_yaxes(title_text="Base Feed Rate (L/h)", row=1, col=2)
fig_compare.update_yaxes(title_text="Total Inlet Rate (L/h)", row=1, col=3)
fig_compare.update_yaxes(title_text="Volume (L)", row=1, col=4)
fig_compare.update_yaxes(title_text="Glucose (mmol/L)", row=1, col=5)

fig_compare.update_layout(
    title_text="Perfusion Strategy Comparison",
    template="plotly_white",
    height=500,
    width=3200,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
)

fig_compare.show()


Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\perfusion.yaml
Successfully loaded parameters from 5 file(s).
Simulating Constant for comparison...
run_si